# Lahore Road Network (OSM)

Fetch OSM road network for Lahore, save to GeoPackage, and plot on a Folium map.


In [5]:
import geopandas as gpd
import osmnx as ox
import folium


In [6]:
# ---- CONFIG ----
BOUNDARY_PATH = "../lahore.geojson"  # Lahore boundary
TARGET_CRS = "EPSG:32643"            # meters (UTM 43N for Lahore)

# Output
OUT_GPKG = "lahore_roads.gpkg"
OUT_LAYER = "roads"
OUT_HTML = "lahore_roads_map.html"

# OSMnx filter: major + minor + residential
CUSTOM_FILTER = (
    '["highway"]["area"!~"yes"]'
    '["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified|residential|service|living_street"]'
)


In [7]:
# ---- LOAD BOUNDARY ----
boundary = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
lahore_boundary = boundary.unary_union
print("Boundary loaded")


Boundary loaded


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_75291/1831077039.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lahore_boundary = boundary.unary_union


In [8]:
# ---- FETCH ROADS ----
# Download roads within Lahore boundary
if hasattr(ox, "geometries_from_polygon"):
    roads = ox.geometries_from_polygon(lahore_boundary, tags={"highway": True})
else:
    roads = ox.features_from_polygon(lahore_boundary, tags={"highway": True})

# Filter to lines only
roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])].copy()

# Optional filter to common highway classes
if "highway" in roads.columns:
    roads = roads[roads["highway"].astype(str).str.contains(
        "motorway|trunk|primary|secondary|tertiary|unclassified|residential|service|living_street",
        regex=True,
        na=False,
    )]

print("Roads:", len(roads))


Roads: 95505


In [9]:
# ---- SAVE ----
roads.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print("Wrote:", OUT_GPKG, OUT_LAYER)


Wrote: lahore_roads.gpkg roads


In [ ]:
# ---- PLOT MAP ----
cent = boundary.geometry.unary_union.centroid
m = folium.Map(location=[cent.y, cent.x], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(boundary, name="Boundary").add_to(m)
folium.GeoJson(roads.to_crs(4326), name="Roads").add_to(m)

folium.LayerControl().add_to(m)

m.save(OUT_HTML)
print("Wrote:", OUT_HTML)

m


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_75291/2213520195.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = boundary.geometry.unary_union.centroid
/Users/ahmed/Library/Python/3.9/lib/python/site-packages/geopandas/geodataframe.py:1046: FutureWarning: Starting with NumPy 2.0, the behavior of the 'copy' keyword has changed and passing 'copy=False' raises an error when returning a zero-copy NumPy array is not possible. pandas will follow this behavior starting with pandas 3.0.
This conversion to NumPy requires a copy, but 'copy=False' was passed. Consider using 'np.asarray(..)' instead.
  ids = np.array(self.index, copy=False)
/Users/ahmed/Library/Python/3.9/lib/python/site-packages/geopandas/geodataframe.py:1046: FutureWarning: Starting with NumPy 2.0, the behavior of the 'copy' keyword has changed and passing 'copy=False' raises an error when returning a zero-copy NumPy array is not possible. pandas will

Wrote: lahore_roads_map.html
